# CoreForge AI Evaluation v2.1 — Hindi QA knowledge comparison

**Research only:** This notebook does not train a model, publish model weights, or change Adatacore production. Compare untouched **Qwen3-4B-Instruct-2507** with/without retrieval on 40 previous benchmark questions plus 12 newly authored held-out questions.

Knowledge references are **draft, pending project-owner review**, not official style-guide policy. The held-out question reference IDs are for scoring retrieval only and are never passed to the model. This remains a diagnostic rather than a blinded independent evaluation because the test authors also authored the references.

Changes: revised filler/NST guidance, hybrid retrieval with domain aliases, output-budget increase from 160 to 256, explicit end token, suspicious-response warnings, resumable progress, and a human scoring CSV.

**Runtime:** Colab T4 GPU (availability not guaranteed). Restart in a fresh runtime for this notebook.


In [ ]:
import torch, shutil
assert torch.cuda.is_available(), "Please select Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Available GPU GiB:",round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
print("Free disk GiB:",round(shutil.disk_usage('/content').free/2**30,1))


## 1 — Install packages and retrieve benchmark files

This notebook does not require Unsloth or any CoreForge training script. It loads a quantized model using Hugging Face Transformers directly. It downloads the two GitHub JSON files so the test uses versioned knowledge and questions.


In [ ]:
%pip -q install 'transformers>=4.57,<5' 'accelerate>=1.8,<2' 'bitsandbytes>=0.46' 'scikit-learn>=1.4,<2' 'safetensors>=0.5'


In [ ]:
import json, hashlib, urllib.request
from pathlib import Path
ROOT=Path('/content/coreforge-eval-v21')
ROOT.mkdir(exist_ok=True)
BASE_URL='https://raw.githubusercontent.com/ajay333X/my-adatacore-site/main/workers/'
for name in ('coreforge_eval_v21_knowledge.json','coreforge_eval_v2_questions.json','coreforge_eval_v21_unseen.json'):
    urllib.request.urlretrieve(BASE_URL+name,ROOT/name)
knowledge=json.loads((ROOT/'coreforge_eval_v21_knowledge.json').read_text(encoding='utf-8'))
old_bank=json.loads((ROOT/'coreforge_eval_v2_questions.json').read_text(encoding='utf-8'))
unseen=json.loads((ROOT/'coreforge_eval_v21_unseen.json').read_text(encoding='utf-8'))
entries=knowledge['entries']
questions=[{**q,'test_split':'reused_v2'} for q in old_bank['questions']]+[
  {**q,'test_split':'new_holdout'} for q in unseen['questions']]
assert len(questions)==52 and len({q['id'] for q in questions})==52 and len(entries)>=12
keys={e['id'] for e in entries}
assert all(set(q['reference_ids']).issubset(keys) for q in questions)
print('Questions:',len(questions),'| Reused:',len(old_bank['questions']),
      '| New:',len(unseen['questions']),'| Knowledge:',len(entries))
print('Reference approval:',knowledge['status'])
print('Human sign-off required before production.')


## 2 — Load the untouched Qwen3-4B Instruct model

4-bit quantization is used to fit common Colab T4 memory limits. If GPU memory is insufficient, do not proceed to training or buy hardware—share the error. We use the model's instructed chat template without a thinking-mode flag.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL_ID='Qwen/Qwen3-4B-Instruct-2507'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,quantization_config=quant,device_map='auto',
    torch_dtype=torch.float16,trust_remote_code=False
)
model.eval()
eos=tokenizer.convert_tokens_to_ids('<|im_end|>')
if not isinstance(eos,int) or eos<0: raise ValueError('Missing Qwen chat-end token; stop and check tokenizer version.')
print('Loaded:',MODEL_ID,'stop token:',eos)


## 3 — Hybrid retrieval (no answer-key leakage)

Retrieval combines **character n-gram TF-IDF**, **word-level TF-IDF**, and small question-language aliases to better match audio-related topics. Unlike an oracle evaluation, it never uses a question's expected reference IDs to choose passages. The expected IDs remain evaluation metadata only. The top three passages are returned, with IDs and similarity scores logged.


In [ ]:
import re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
ALIASES={
 'fillers':['उम्','अह','फ़िलर','फिलर','filler','आवाज़','बोली गई ध्वनि'],
 'audio_nst':['हँसी','साँस','नॉन स्पीच','non speech','ध्वनि टैग','nst'],
 'word_accuracy':['ऑडियो','शब्द','रिकॉर्डिंग','सटीकता','ट्रांसक्रिप्शन'],
 'punctuation':['ठहराव','विराम','पूर्ण विराम','pause','विराम टैग'],
 'loanwords':['ऋणशब्द','loanword','foreign-language','ईमेल','फ़ोन'],
 'wer':['word error rate','wer','त्रुटि दर','शब्द त्रुटि'],
 'cer':['character error rate','cer','अक्षर त्रुटि']
}
documents=[e['title']+'\n'+e['content']+'\n'+' '.join(ALIASES.get(e['id'],[])) for e in entries]
charvec=TfidfVectorizer(analyzer='char',ngram_range=(2,4),sublinear_tf=True)
wordvec=TfidfVectorizer(analyzer='word',token_pattern=r'(?u)\b\w+\b',ngram_range=(1,2),sublinear_tf=True)
cmat=charvec.fit_transform(documents)
wmat=wordvec.fit_transform(documents)
def retrieve(prompt,k=3):
    charsim=cosine_similarity(charvec.transform([prompt]),cmat).ravel()
    wordsim=cosine_similarity(wordvec.transform([prompt]),wmat).ravel()
    scores=.75*charsim+.25*wordsim
    ranked=sorted(range(len(entries)),key=lambda i:(-float(scores[i]),i))[:k]
    return [{**entries[i],'similarity':round(float(scores[i]),4)} for i in ranked]
for q in questions[:4]+questions[-4:]:
    docs=retrieve(q['prompt'])
    print(q['id'],q['test_split'],'retrieved:',[(x['id'],x['similarity']) for x in docs])


## 4 — Paired generation: 52 × two conditions

Default is all **52 questions = 104 responses**. Set `QUESTION_LIMIT=5` for a quick smoke test. Using longer **256-token** answers reduces, but cannot eliminate, incomplete responses. Each result is saved to a partial JSON file after the pair finishes, so you can rerun in the same Colab session without repeating completed pairs. No answer-key documents are supplied to retrieval.

Heuristic warning checks flag potential misconceptions (e.g. translating a spoken filler or blanket approval to change words without audio). They are *review flags*, not automatic adjudications. All responses must be human-checked.


In [ ]:
import re, time, collections
QUESTION_LIMIT=52
assert 1<=QUESTION_LIMIT<=52
OUT=ROOT/'coreforge-eval-v21-partial.json'
SYSTEM="आप एक हिंदी QA सहायक हैं। केवल सवाल का संक्षिप्त, तथ्यात्मक उत्तर दें (अधिकतम दो छोटे वाक्य)। किसी जानकारी की पुष्टि न हो तो साफ कहें; तथ्य न गढ़ें।"
CONTEXT_RULES="संदर्भ जानकारी केवल तथ्य-जाँच के लिए है। अगर संदर्भ अपूर्ण है तो न गढ़ें। संदर्भ के बाहर प्रमाणित जानकारी न हो तो अनिश्चितता बताएं।"
def make_chat(question,docs=None):
    sysmsg=SYSTEM
    if docs:
        excerpts='\n\n'.join(f"[{d['id']}] {d['title']}: {d['content']}" for d in docs)
        sysmsg+='\n'+CONTEXT_RULES+'\nसंदर्भ:\n'+excerpts
    return tokenizer.apply_chat_template(
        [{"role":"system","content":sysmsg},{"role":"user","content":question}],
        tokenize=False,add_generation_prompt=True)
def repetition_ratio(answer):
    tokens=re.findall(r"\S+",answer.lower())
    if len(tokens)<7:return 0.0
    grams=[tuple(tokens[i:i+3]) for i in range(len(tokens)-2)]
    return round(1.0-len(set(grams))/len(grams),3)
def generate_answer(question,docs,seed):
    torch.manual_seed(seed)
    prompt=make_chat(question,docs)
    encoded=tokenizer(prompt,return_tensors='pt')
    encoded={k:v.to(model.device) for k,v in encoded.items()}
    with torch.inference_mode():
        generated=model.generate(**encoded,max_new_tokens=256,
            do_sample=True,temperature=0.7,top_p=0.8,top_k=20,
            eos_token_id=eos,pad_token_id=eos)
    ids=generated[0][encoded['input_ids'].shape[-1]:].tolist()
    text=tokenizer.decode(ids,skip_special_tokens=True).strip()
    return {"answer":text,"tokens":len(ids),"stopped_on_eos":bool(ids and ids[-1]==eos),
        "hit_limit":bool(len(ids)>=256 and ids[-1]!=eos),
        "replacement_characters":text.count("\ufffd"),
        "repetition_ratio":repetition_ratio(text)}
def heuristic_warnings(question,answer,metrics):
    flags=[]
    if metrics['hit_limit']:flags.append('possibly_truncated')
    if metrics['replacement_characters']:flags.append('text_replacement_characters')
    if metrics['repetition_ratio']>0.22:flags.append('possible_repetition')
    lower=answer.casefold()
    q=question.casefold()
    if ('उम्' in q or 'अह' in q or 'filler' in q) and (
        'अनुवादित करें' in lower or 'अनुवाद करना' in lower or 'नॉन-स्पीच टैग के रूप' in lower
    ): flags.append('possible_filler_misclassification')
    if ('बिना ऑडियो' in q or 'सुने बिना' in q) and ('बदलना चाहिए।' in lower or 'हटा सकते हैं' in lower):
        flags.append('possible_unauthorized_change')
    return flags
if OUT.exists():
    old=json.loads(OUT.read_text(encoding='utf-8'))
    assert old.get('model')==MODEL_ID and old.get('version')==knowledge['version'], "Partial results belong to another experiment."
    results=old['results']
else:
    results=[]
done={r['id'] for r in results}
for i,q in enumerate(questions[:QUESTION_LIMIT]):
    if q['id'] in done: continue
    refs=retrieve(q['prompt'],k=3)
    base=generate_answer(q['prompt'],None,20261113+i)
    grounded=generate_answer(q['prompt'],refs,20261113+i)
    result={"id":q['id'],"category":q['category'],"test_split":q['test_split'],"prompt":q['prompt'],
        "reference_ids_for_review":q['reference_ids'],
        "retrieved_ids":[r['id'] for r in refs],
        "retrieval_similarities":[r['similarity'] for r in refs],
        "no_context":base,"retrieved_context":grounded}
    result['warnings']={key:heuristic_warnings(q['prompt'],result[key]['answer'],result[key]) for key in ('no_context','retrieved_context')}
    results.append(result)
    OUT.write_text(json.dumps({"version":knowledge['version'],"model":MODEL_ID,"results":results,
        "research_only":True,"context_type":"hybrid_tfidf_retrieval","production_approved":False},ensure_ascii=False,indent=2),encoding='utf-8')
    print(f"{len(results)}/{QUESTION_LIMIT} {q['id']} | bare: {base['answer'][:100]} | grounded: {grounded['answer'][:100]}",flush=True)
print("Paired comparisons saved:",len(results),"at",OUT)


## 5 — Report + human-review CSV

Summary includes retrieval hit rate, truncated-answer counts, replacement characters, possible contradictions and split-specific counts. These are *diagnostic* metrics, **not factual accuracy scores**. Human review of both 40 reused and 12 unseen questions remains essential. The candidate knowledge has not been officially approved.


In [ ]:
import csv, statistics
from google.colab import files
complete=len(results)==len(questions)
def summarize(key):
    vals=[r[key] for r in results]
    return {"answers":len(vals),
        "hit_limit":sum(bool(x['hit_limit']) for x in vals),
        "stopped_on_eos":sum(bool(x['stopped_on_eos']) for x in vals),
        "with_replacement_characters":sum(x['replacement_characters']>0 for x in vals),
        "high_repetition":sum(x['repetition_ratio']>0.22 for x in vals),
        "avg_output_tokens":round(statistics.mean(x['tokens'] for x in vals),2) if vals else None}
hits=sum(bool(set(r['retrieved_ids'])&set(r['reference_ids_for_review'])) for r in results)
report={"version":knowledge['version'],"model":MODEL_ID,"completed_pairs":len(results),
        "total_questions":len(questions),"complete":complete,
        "context_type":"hybrid_automatically_retrieved_candidate_reference_text",
        "knowledge_status":knowledge['status'],
        "knowledge_sha256":hashlib.sha256((ROOT/'coreforge_eval_v21_knowledge.json').read_bytes()).hexdigest(),
        "question_bank_sha256":hashlib.sha256((ROOT/'coreforge_eval_v2_questions.json').read_bytes()).hexdigest(),
        "generation":{"max_new_tokens":256,"temperature":0.7,"top_p":0.8,"top_k":20},
        "retrieval":{"any_expected_reference_in_top3":hits,"rate":round(hits/len(results),3) if results else None},
        "no_context":summarize('no_context'),"retrieved_context":summarize('retrieved_context'),
        "production_approved":False,"human_quality_assessment":"pending","results":results,
        "test_splits":{"reused_v2":sum(r['test_split']=='reused_v2' for r in results),
                       "new_holdout":sum(r['test_split']=='new_holdout' for r in results)},
        "warning_counts":{mode:sum(bool(r.get('warnings',{}).get(mode)) for r in results) for mode in ('no_context','retrieved_context')}}
FINAL=ROOT/'coreforge-eval-v21-results.json'
REVIEW=ROOT/'coreforge-eval-v21-human-review.csv'
FINAL.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
with REVIEW.open('w',encoding='utf-8',newline='') as f:
    w=csv.writer(f)
    w.writerow(["id","test_split","prompt","expected_references","retrieved_references","no_context_answer","grounded_answer",
      "no_context_accuracy_0_2","grounded_accuracy_0_2","no_context_hindi_0_2","grounded_hindi_0_2",
      "no_context_instruction_0_2","grounded_instruction_0_2","human_notes","no_context_flags","grounded_flags"])
    for r in results:
        w.writerow([r['id'],r['test_split'],r['prompt'],','.join(r['reference_ids_for_review']),','.join(r['retrieved_ids']),
            r['no_context']['answer'],r['retrieved_context']['answer'],"","","","","","","",",".join(r.get('warnings',{}).get('no_context',[])),",".join(r.get('warnings',{}).get('retrieved_context',[]))])
print("Complete:",complete,"| Retrieval hits:",hits,"/",len(results))
print("No context:",report['no_context'])
print("Retrieved context:",report['retrieved_context'])
files.download(str(FINAL))
files.download(str(REVIEW))


## Deliverables
Upload **coreforge-eval-v21-results.json** and optionally the scored **coreforge-eval-v21-human-review.csv**. Compare improvements and regressions, especially H07/H35 and U01/U06 fillers and the new withheld questions. **Do not deploy a production QA assistant until references and model behavior are reviewed and approved.**
